<a href="https://colab.research.google.com/github/ayishathanha53-eng/AI-ML-lab/blob/main/id3%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [ ]:
def entropy(target_col):
    """
    Calculate the entropy of a list/series of observations.
    """
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = np.sum([(-counts[i] / np.sum(counts)) * np.log2(counts[i] / np.sum(counts))
                          for i in range(len(elements))])
    return entropy_val

In [ ]:
def information_gain(data, split_attribute_name, target_name="target"):
    """
    Calculate the information gain of a given attribute.
    """
    # Calculate the entropy of the total dataset
    total_entropy = entropy(data[target_name])

    # Calculate the values and counts for the split attribute
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)

    # Calculate the weighted entropy
    weighted_entropy = np.sum([(counts[i] / np.sum(counts)) * entropy(data.where(data[split_attribute_name] == vals[i]).dropna()[target_name])
                                 for i in range(len(vals))])

    # Calculate information gain
    ig = total_entropy - weighted_entropy
    return ig

In [ ]:
def split_data(data, split_attribute_name):
    """
    Split the dataset based on the unique values of a given attribute.
    Returns a dictionary of dataframes, where keys are the unique values
    of the splitting attribute.
    """
    unique_values = np.unique(data[split_attribute_name])
    split_datasets = {val: data[data[split_attribute_name] == val].drop(columns=[split_attribute_name])
                      for val in unique_values}
    return split_datasets

In [ ]:
def id3(data, original_data, features, target_attribute_name="target", parent_node_class=None):
    """
    ID3 algorithm to build a decision tree.
    """
    # Define the stopping criteria

    # 1. If all target_attribute_name values are the same, return that value
    if len(np.unique(data[target_attribute_name])) == 1:
        return np.unique(data[target_attribute_name])[0]

    # 2. If the dataset is empty, return the parent_node_class
    elif len(data) == 0:
        return parent_node_class

    # 3. If there are no more features to split on, return the most common class
    elif len(features) == 0:
        return np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

    # Otherwise, continue to build the tree
    else:
        # Determine the parent_node_class for the current node
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

        # Select the feature that best splits the data (highest information gain)
        item_values = [information_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create the tree structure
        tree = {best_feature: {}}

        # Remove the best feature from the feature list for the next recursion
        features = [i for i in features if i != best_feature]

        # Recursively build the tree for each unique value of the best feature
        for value in np.unique(data[best_feature]):
            value = value
            # Split the data based on the best feature's values
            sub_data = data.where(data[best_feature] == value).dropna()

            # Recursively call ID3 for the sub-dataset
            subtree = id3(sub_data, original_data, features, target_attribute_name, parent_node_class)

            # Add the subtree to the main tree
            tree[best_feature][value] = subtree

        return tree

In [ ]:
def predict(query, tree):
    """
    Predicts the target class for a new query using the trained decision tree.
    """
    # Traverse the tree
    for key in list(query.keys()):
        if key in tree:
            try:
                value = query[key]
                tree = tree[key][value]
            except KeyError:
                return None # Handle cases where a branch for a value doesn't exist

            if isinstance(tree, dict):
                return predict(query, tree)
            else:
                return tree
    return tree # Should only reach here if tree is already a leaf node


In [ ]:
# Sample Dataset (e.g., whether to play tennis)
data = pd.DataFrame({
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
})

# Rename 'PlayTennis' to 'target' for consistency with our functions
data = data.rename(columns={'PlayTennis': 'target'})
display(data.head())

,Outlook,Temperature,Humidity,Wind,target
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


In [ ]:
# Get the list of features (attributes excluding the target)
features = list(data.columns[:-1])

# Build the ID3 decision tree
tree = id3(data, data, features, target_attribute_name='target')

print("Generated Decision Tree:")
import json
print(json.dumps(tree, indent=4))

Generated Decision Tree:
{
    "Outlook": {
        "Overcast": "Yes",
        "Rain": {
            "Wind": {
                "Strong": "No",
                "Weak": "Yes"
            }
        },
        "Sunny": {
            "Humidity": {
                "High": "No",
                "Normal": "Yes"
            }
        }
    }
}


In [ ]:
# Make a prediction for a new instance
query = {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Wind': 'Weak'}
prediction = predict(query, tree)

print(f"\nPredicting for: {query}")
print(f"Prediction: {prediction}")

query2 = {'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'}
prediction2 = predict(query2, tree)

print(f"\nPredicting for: {query2}")
print(f"Prediction: {prediction2}")


Predicting for: {'Outlook': 'Sunny', 'Temperature': 'Hot', 'Humidity': 'Normal', 'Wind': 'Weak'}
Prediction: Yes

Predicting for: {'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'}
Prediction: Yes
